# Step 3 — diagnose block topology before merging

**# of cells in notebook:** 2 code cells

## Purpose

Evaluate whether small geometry gaps, overlaps, point-only contacts, or coordinate precision are fragmenting the block adjacency network that will be used for block merging. The notebook builds the same general type of same-zone rook-adjacency network needed by the merging workflow, identifies topology anomalies and disconnected components, and tests several candidate coordinate-precision grids before any precision adjustment is applied during the actual merge.

The source `blocks_5.gpkg` is not modified.

## Input

**Prepared blocks**
- GeoPackage: `E:\_johannesburg\_analysis\segments_v2\blocks_5.gpkg`
- Layer: `blocks_5`

Key fields include:

- `block_id`
- `zones_5_ID`
- `zones_5_pop`
- `population`
- `open_space`
- `airport`

## Output

**Primary diagnostic folder**
- `E:\_johannesburg\_analysis\segments_v2\topology_diagnostics`

**Primary GeoPackage**
- `blocks_5_topology_diagnostics.gpkg`
- Spatial layer: `blocks_topology_diagnostic`

Diagnostic CSVs include:

- `invalid_geometry.csv`
- `city_summary.csv`
- `zone_summary.csv`
- `component_summary_raw.csv`
- `topology_anomalies.csv`
- `precision_summary.csv`
- `zone_precision_summary.csv`
- `precision_edge_changes.csv`

Where supported, the summary tables are also written as nonspatial tables in the diagnostic GeoPackage.

The block-ID QA cell additionally writes CSV reports for null, blank, or duplicated `block_id` values.

## Main logic

### Cell 1 — diagnose topology and test coordinate precision

1. Read `blocks_5` and confirm that it uses a projected coordinate system.
2. Validate required identifiers, population fields, exclusion flags, and polygon geometry.
3. Treat blocks flagged as either `open_space = 1` or `airport = 1` as merge-excluded.
4. Define all remaining blocks as eligible for the block-merging topology analysis.
5. Require rook adjacency to consist of a shared polygon boundary longer than `0.05 m`.
6. Build the raw same-zone rook graph for merge-eligible blocks.

#### Topology-anomaly diagnostics

7. Examine intersecting polygon pairs and identify:
   - valid rook adjacency;
   - point-only contacts;
   - tiny overlaps of `0.01 m²` or less;
   - larger/material overlaps.
8. Search for close same-zone polygon pairs separated by gaps of up to `0.25 m`.
9. Record topology anomalies and the affected block pairs.

#### Connected components and islands

10. Identify connected components in the raw rook graph.
11. Identify degree-zero blocks that are topological islands.
12. Calculate the total population of each connected component.
13. Flag components with total population below 500, since no within-component merge can raise such a component to the 500-person target without connecting it to another component.

#### Coordinate-precision tests

14. Test the topology after applying candidate coordinate grids of:
    - `0.01 m`;
    - `0.05 m`;
    - `0.10 m`.
15. For each candidate grid, compare the result with the raw geometry in terms of:
    - rook-edge count;
    - new adjacency edges;
    - lost adjacency edges;
    - connected components;
    - island blocks;
    - geometry validity;
    - polygon-area change.
16. Record the specific block pairs responsible for new or lost edges.
17. Create citywide, zone-level, component-level, and block-level diagnostic outputs.

The principal files to review when selecting a working precision are `precision_summary.csv` and `precision_edge_changes.csv`. A suitable precision grid should resolve microscopic topology artifacts while creating few or no inappropriate adjacency changes and causing negligible geometry-area change.

### Cell 2 — verify `block_id` integrity

1. Read the `block_id` field while retaining the GeoPackage feature ID so any problem feature can be located.
2. Identify null or blank `block_id` values.
3. Identify duplicate `block_id` values.
4. Write detailed and summarized CSV reports of any problems.

This provides an additional identifier QA check before the block-merging workflow is run.


In [ ]:
"""
Citywide topology diagnostics for the block-merging workflow.

Purpose
-------
This script does NOT modify the source GeoPackage. It diagnoses whether tiny
geometry gaps, overlaps, or point-only contacts are fragmenting the strict
same-zone rook graph, and compares the graph after several candidate precision
(grid-snapping) settings.

Primary outputs
---------------
1. blocks_5_topology_diagnostics.gpkg
   - blocks_topology_diagnostic: source blocks plus topology flags
   - nonspatial summary tables, where supported by the installed GDAL build
2. CSV files in OUT_FOLDER:
   - city_summary.csv
   - zone_summary.csv
   - component_summary_raw.csv
   - topology_anomalies.csv
   - precision_summary.csv
   - zone_precision_summary.csv
   - precision_edge_changes.csv
   - invalid_geometry.csv

Interpretation
--------------
A candidate precision grid is promising when it removes obvious artificial
islands/components, creates few or no lost rook edges, changes polygon areas
only negligibly, and its new edges correspond to microscopic overlaps/gaps or
point contacts rather than substantive separations.
"""

from __future__ import annotations

import math
import os
import re
import traceback
from collections import defaultdict

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import shapely


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

IN_GPKG = r"E:\_johannesburg\_analysis\segments_v2\blocks_5.gpkg"
IN_LAYER = "blocks_5"  # ArcGIS may display this as main.blocks_5

OUT_FOLDER = (
    r"E:\_johannesburg\_analysis\segments_v2"
    r"\topology_diagnostics"
)
OUT_GPKG = os.path.join(OUT_FOLDER, "blocks_5_topology_diagnostics.gpkg")


# ---------------------------------------------------------------------------
# Fields
# ---------------------------------------------------------------------------

BLOCK_ID = "block_id"
ZONE_ID = "zones_5_ID"
ZONE_POP = "zones_5_pop"
POP = "population"
OPEN_SPACE = "open_space"
AIRPORT = "airport"


# ---------------------------------------------------------------------------
# Diagnostic parameters
# ---------------------------------------------------------------------------

# Strict rook adjacency requires a shared line longer than this.
MIN_SHARED_EDGE_M = 0.05

# Pair-classification thresholds.
OVERLAP_EPS_M2 = 1e-9
TINY_OVERLAP_M2 = 0.01
NEAR_GAP_MAX_M = 0.25

# Compare strict topology after rounding coordinates to these grids.
# None/raw is always tested separately.
PRECISION_GRIDS_M = [0.01, 0.05, 0.10]

# Prevent an unexpectedly huge near-gap CSV. Counts are still retained even
# after the row-writing cap is reached.
MAX_NEAR_GAP_ROWS = 250_000

USE_ARROW = False


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def log(message: str) -> None:
    print(message, flush=True)


def resolve_layer(path: str, requested: str) -> str:
    requested = requested.split(".")[-1]
    names = [str(row[0]) for row in pyogrio.list_layers(path)]
    for name in names:
        if name.lower() == requested.lower():
            return name
    raise ValueError(f"Layer '{requested}' not found. Available layers: {names}")


def resolve_field(columns, requested: str) -> str:
    lookup = {str(c).lower(): str(c) for c in columns}
    try:
        return lookup[requested.lower()]
    except KeyError as exc:
        raise ValueError(
            f"Missing field '{requested}'. Available fields:\n{list(columns)}"
        ) from exc


def safe_zone_text(value) -> str:
    if pd.isna(value):
        return "NULL"
    text = str(value)
    if text.endswith(".0"):
        text = text[:-2]
    text = re.sub(r"[^0-9A-Za-z_-]+", "_", text)
    return text or "EMPTY"


def grid_tag(grid: float) -> str:
    return str(grid).replace(".", "p")


def normalize_open_space(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    bad = ~values.isin([0, 1])
    if bad.any():
        examples = sorted(values.loc[bad].unique().tolist())[:10]
        raise ValueError(
            "open_space must contain only 0, 1, or null. "
            f"Unexpected values include: {examples}"
        )
    return values.eq(1)


def normalize_airport(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    bad = ~values.isin([0, 1])
    if bad.any():
        examples = sorted(values.loc[bad].unique().tolist())[:10]
        raise ValueError(
            "airport must contain only 0, 1, or null. "
            f"Unexpected values include: {examples}"
        )
    return values.eq(1)


def connected_components(graph: dict[int, dict[int, float]]) -> list[list[int]]:
    unseen = set(graph)
    result: list[list[int]] = []

    while unseen:
        start = min(unseen)
        unseen.remove(start)
        stack = [start]
        comp = []

        while stack:
            i = stack.pop()
            comp.append(i)
            for j in graph[i]:
                if j in unseen:
                    unseen.remove(j)
                    stack.append(j)

        result.append(sorted(comp))

    return sorted(result, key=lambda members: min(members))


def assign_component_ids(
    graph: dict[int, dict[int, float]],
    zone_values: np.ndarray,
) -> tuple[dict[int, str], list[list[int]]]:
    assignment: dict[int, str] = {}
    all_components: list[list[int]] = []

    zones = sorted({zone_values[i] for i in graph}, key=lambda x: str(x))
    for zone in zones:
        nodes = [i for i in graph if zone_values[i] == zone]
        node_set = set(nodes)
        subgraph = {
            i: {j: length for j, length in graph[i].items() if j in node_set}
            for i in nodes
        }
        comps = connected_components(subgraph)
        for number, members in enumerate(comps, start=1):
            cid = f"z{safe_zone_text(zone)}_c{number:03d}"
            for i in members:
                assignment[i] = cid
            all_components.append(members)

    return assignment, all_components


def build_raw_topology(
    gdf: gpd.GeoDataFrame,
    geoms: np.ndarray,
    zone_values: np.ndarray,
    eligible: np.ndarray,
    open_flags: np.ndarray,
    block_ids: np.ndarray,
):
    """Build raw rook graphs and identify intersecting-pair anomalies."""

    n = len(gdf)
    all_graph = {i: {} for i in range(n)}
    eligible_graph = {i: {} for i in range(n) if eligible[i]}
    eligible_edges: set[tuple[int, int]] = set()

    point_contact_count = np.zeros(n, dtype=np.int64)
    overlap_count = np.zeros(n, dtype=np.int64)

    boundaries = np.asarray(shapely.boundary(geoms), dtype=object)
    series = gpd.GeoSeries(geoms, crs=gdf.crs)
    sindex = series.sindex

    anomaly_rows: list[dict] = []

    log("Scanning intersecting polygon pairs...")
    for i, geom in enumerate(geoms):
        if i and i % 2_000 == 0:
            log(f"  intersect scan: {i:,} / {n:,}")

        for j0 in sindex.query(geom, predicate="intersects"):
            j = int(j0)
            if j <= i:
                continue

            shared_geom = shapely.intersection(boundaries[i], boundaries[j])
            shared_len = float(shapely.length(shared_geom))

            if shared_len > MIN_SHARED_EDGE_M:
                all_graph[i][j] = shared_len
                all_graph[j][i] = shared_len

            same_zone_eligible = (
                eligible[i]
                and eligible[j]
                and zone_values[i] == zone_values[j]
            )
            if not same_zone_eligible:
                continue

            inter_geom = shapely.intersection(geom, geoms[j])
            inter_area = float(shapely.area(inter_geom))
            min_area = max(
                1e-12,
                min(float(shapely.area(geom)), float(shapely.area(geoms[j]))),
            )
            overlap_ratio = inter_area / min_area

            if shared_len > MIN_SHARED_EDGE_M:
                eligible_graph[i][j] = shared_len
                eligible_graph[j][i] = shared_len
                eligible_edges.add((i, j))

            if inter_area > OVERLAP_EPS_M2:
                pair_type = (
                    "tiny_overlap"
                    if inter_area <= TINY_OVERLAP_M2
                    else "material_overlap"
                )
                overlap_count[i] += 1
                overlap_count[j] += 1
                anomaly_rows.append(
                    {
                        "pair_type": pair_type,
                        "source_row_ix_a": i,
                        "source_row_ix_b": j,
                        "block_id_a": block_ids[i],
                        "block_id_b": block_ids[j],
                        "zone_id": zone_values[i],
                        "shared_boundary_m": shared_len,
                        "distance_m": 0.0,
                        "intersection_area_m2": inter_area,
                        "overlap_fraction_smaller_polygon": overlap_ratio,
                        "boundary_intersection_type": shared_geom.geom_type,
                    }
                )
            elif shared_len <= MIN_SHARED_EDGE_M:
                point_contact_count[i] += 1
                point_contact_count[j] += 1
                anomaly_rows.append(
                    {
                        "pair_type": "point_only_contact",
                        "source_row_ix_a": i,
                        "source_row_ix_b": j,
                        "block_id_a": block_ids[i],
                        "block_id_b": block_ids[j],
                        "zone_id": zone_values[i],
                        "shared_boundary_m": shared_len,
                        "distance_m": 0.0,
                        "intersection_area_m2": inter_area,
                        "overlap_fraction_smaller_polygon": 0.0,
                        "boundary_intersection_type": shared_geom.geom_type,
                    }
                )

    anomalies = pd.DataFrame(anomaly_rows)
    return (
        all_graph,
        eligible_graph,
        eligible_edges,
        anomalies,
        point_contact_count,
        overlap_count,
    )


def scan_near_gaps(
    gdf: gpd.GeoDataFrame,
    geoms: np.ndarray,
    zone_values: np.ndarray,
    eligible: np.ndarray,
    block_ids: np.ndarray,
):
    n = len(gdf)
    near_gap_count = np.zeros(n, dtype=np.int64)
    nearest_gap = np.full(n, np.nan, dtype=float)
    rows: list[dict] = []
    total_near_gap_pairs = 0

    log(f"Scanning same-zone gaps <= {NEAR_GAP_MAX_M} m...")
    zones = sorted(set(zone_values[eligible]), key=lambda x: str(x))

    for zone_no, zone in enumerate(zones, start=1):
        idx = np.flatnonzero(eligible & (zone_values == zone))
        if len(idx) < 2:
            continue

        local_geoms = geoms[idx]
        series = gpd.GeoSeries(local_geoms, crs=gdf.crs)
        sindex = series.sindex
        has_dwithin = "dwithin" in sindex.valid_query_predicates

        for li, geom in enumerate(local_geoms):
            if has_dwithin:
                candidates = sindex.query(
                    geom,
                    predicate="dwithin",
                    distance=NEAR_GAP_MAX_M,
                )
            else:
                candidates = sindex.query(
                    shapely.buffer(geom, NEAR_GAP_MAX_M),
                    predicate="intersects",
                )

            for lj0 in candidates:
                lj = int(lj0)
                if lj <= li:
                    continue
                other = local_geoms[lj]
                if shapely.intersects(geom, other):
                    continue

                distance = float(shapely.distance(geom, other))
                if not (0.0 < distance <= NEAR_GAP_MAX_M):
                    continue

                i, j = int(idx[li]), int(idx[lj])
                total_near_gap_pairs += 1
                near_gap_count[i] += 1
                near_gap_count[j] += 1
                nearest_gap[i] = (
                    distance
                    if np.isnan(nearest_gap[i])
                    else min(nearest_gap[i], distance)
                )
                nearest_gap[j] = (
                    distance
                    if np.isnan(nearest_gap[j])
                    else min(nearest_gap[j], distance)
                )

                if len(rows) < MAX_NEAR_GAP_ROWS:
                    rows.append(
                        {
                            "pair_type": "near_gap",
                            "source_row_ix_a": i,
                            "source_row_ix_b": j,
                            "block_id_a": block_ids[i],
                            "block_id_b": block_ids[j],
                            "zone_id": zone,
                            "shared_boundary_m": 0.0,
                            "distance_m": distance,
                            "intersection_area_m2": 0.0,
                            "overlap_fraction_smaller_polygon": 0.0,
                            "boundary_intersection_type": None,
                        }
                    )

        if zone_no % 20 == 0:
            log(f"  near-gap scan: {zone_no:,} / {len(zones):,} zones")

    return pd.DataFrame(rows), near_gap_count, nearest_gap, total_near_gap_pairs


def build_eligible_rook_graph(
    gdf: gpd.GeoDataFrame,
    geoms: np.ndarray,
    zone_values: np.ndarray,
    eligible: np.ndarray,
):
    graph = {i: {} for i in range(len(gdf)) if eligible[i]}
    edges: set[tuple[int, int]] = set()
    boundaries = np.asarray(shapely.boundary(geoms), dtype=object)
    series = gpd.GeoSeries(geoms, crs=gdf.crs)
    sindex = series.sindex

    for i in graph:
        geom = geoms[i]
        if shapely.is_empty(geom):
            continue
        for j0 in sindex.query(geom, predicate="intersects"):
            j = int(j0)
            if j <= i or not eligible[j] or zone_values[i] != zone_values[j]:
                continue
            shared = float(
                shapely.length(
                    shapely.intersection(boundaries[i], boundaries[j])
                )
            )
            if shared > MIN_SHARED_EDGE_M:
                graph[i][j] = shared
                graph[j][i] = shared
                edges.add((i, j))

    return graph, edges


def island_reason(
    i: int,
    all_graph: dict[int, dict[int, float]],
    zone_values: np.ndarray,
    merge_excluded_flags: np.ndarray,
) -> str:
    neighbors = list(all_graph[i])
    if not neighbors:
        return "no_rook_neighbor_anywhere"

    same_zone_excluded = any(
        zone_values[j] == zone_values[i] and merge_excluded_flags[j]
        for j in neighbors
    )
    other_zone = any(zone_values[j] != zone_values[i] for j in neighbors)
    same_zone_eligible = any(
        zone_values[j] == zone_values[i] and not merge_excluded_flags[j]
        for j in neighbors
    )

    parts = []
    if same_zone_excluded:
        parts.append("rook_only_to_merge_excluded")
    if other_zone:
        parts.append("rook_to_other_zone")
    if same_zone_eligible:
        parts.append("unexpected_same_zone_eligible")
    return "+".join(parts) if parts else "no_eligible_same_zone_rook_neighbor"


def write_table(df: pd.DataFrame, csv_name: str, gpkg_layer: str) -> None:
    csv_path = os.path.join(OUT_FOLDER, csv_name)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    if len(df):
        try:
            pyogrio.write_dataframe(
                df,
                OUT_GPKG,
                layer=gpkg_layer,
                driver="GPKG",
                use_arrow=USE_ARROW,
            )
        except Exception as exc:
            log(
                f"Warning: could not write nonspatial GPKG table "
                f"'{gpkg_layer}': {exc}"
            )


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


def main() -> None:
    os.makedirs(OUT_FOLDER, exist_ok=True)

    layer = resolve_layer(IN_GPKG, IN_LAYER)
    log(f"Reading {IN_GPKG} | layer={layer}")
    gdf = pyogrio.read_dataframe(
        IN_GPKG,
        layer=layer,
        use_arrow=USE_ARROW,
        force_2d=True,
    ).reset_index(drop=True)

    if gdf.crs is None or not gdf.crs.is_projected:
        raise ValueError("Input must use a projected CRS, such as UTM 35S.")

    block_col = resolve_field(gdf.columns, BLOCK_ID)
    zone_col = resolve_field(gdf.columns, ZONE_ID)
    zone_pop_col = resolve_field(gdf.columns, ZONE_POP)
    pop_col = resolve_field(gdf.columns, POP)
    open_col = resolve_field(gdf.columns, OPEN_SPACE)
    airport_col = resolve_field(gdf.columns, AIRPORT)

    gdf["source_row_ix"] = np.arange(len(gdf), dtype=np.int64)

    if gdf[block_col].isna().any() or gdf[block_col].duplicated().any():
        raise ValueError("block_id must be non-null and unique.")
    if gdf[zone_col].isna().any():
        raise ValueError("zones_5_ID contains null values.")

    population = pd.to_numeric(gdf[pop_col], errors="raise").to_numpy(float)
    zone_pop = pd.to_numeric(gdf[zone_pop_col], errors="coerce").to_numpy(float)
    if np.isnan(population).any() or (population < 0).any():
        raise ValueError("population must be non-null and >= 0. Zero is allowed.")

    open_flags = normalize_open_space(gdf[open_col]).to_numpy(bool)
    airport_flags = normalize_airport(gdf[airport_col]).to_numpy(bool)
    merge_excluded_flags = open_flags | airport_flags
    eligible = ~merge_excluded_flags
    zone_values = gdf[zone_col].to_numpy()
    block_ids = gdf[block_col].astype(str).to_numpy()
    geoms = gdf.geometry.to_numpy()

    null_geom = gdf.geometry.isna().to_numpy()
    empty_geom = shapely.is_empty(geoms)
    valid_geom = shapely.is_valid(geoms)
    valid_reason = shapely.is_valid_reason(geoms)

    invalid_rows = pd.DataFrame(
        {
            "source_row_ix": gdf["source_row_ix"],
            "block_id": block_ids,
            "zone_id": zone_values,
            "is_null": null_geom,
            "is_empty": empty_geom,
            "is_valid": valid_geom,
            "validity_reason": valid_reason,
        }
    )
    invalid_rows = invalid_rows[
        invalid_rows["is_null"]
        | invalid_rows["is_empty"]
        | ~invalid_rows["is_valid"]
    ].copy()
    invalid_rows.to_csv(
        os.path.join(OUT_FOLDER, "invalid_geometry.csv"),
        index=False,
        encoding="utf-8-sig",
    )

    if len(invalid_rows):
        raise ValueError(
            f"Found {len(invalid_rows):,} null, empty, or invalid geometries. "
            "See invalid_geometry.csv. Repair these before topology testing."
        )

    log(f"Blocks: {len(gdf):,}")
    log(f"Zones: {gdf[zone_col].nunique():,}")
    log(f"Open-space blocks: {open_flags.sum():,}")
    log(f"Airport blocks: {airport_flags.sum():,}")
    log(f"Merge-excluded blocks: {merge_excluded_flags.sum():,}")
    log(f"Eligible merge blocks: {eligible.sum():,}")
    log(f"Zero-population eligible blocks: {(eligible & (population == 0)).sum():,}")

    (
        all_graph,
        raw_graph,
        raw_edges,
        intersect_anomalies,
        point_contact_count,
        overlap_count,
    ) = build_raw_topology(
        gdf,
        geoms,
        zone_values,
        eligible,
        open_flags,
        block_ids,
    )

    (
        gap_anomalies,
        near_gap_count,
        nearest_gap,
        total_near_gap_pairs,
    ) = scan_near_gaps(
        gdf,
        geoms,
        zone_values,
        eligible,
        block_ids,
    )

    anomalies = pd.concat(
        [intersect_anomalies, gap_anomalies],
        ignore_index=True,
        sort=False,
    )

    raw_assignment, raw_components = assign_component_ids(
        raw_graph, zone_values
    )
    raw_degree = np.zeros(len(gdf), dtype=np.int64)
    for i in raw_graph:
        raw_degree[i] = len(raw_graph[i])

    island_mask = eligible & (raw_degree == 0)
    island_reasons = np.full(len(gdf), None, dtype=object)
    for i in np.flatnonzero(island_mask):
        island_reasons[i] = island_reason(
            i, all_graph, zone_values, merge_excluded_flags
        )

    component_records = []
    for members in raw_components:
        cid = raw_assignment[members[0]]
        zone = zone_values[members[0]]
        comp_pop = float(population[members].sum())
        component_records.append(
            {
                "component_id": cid,
                "zone_id": zone,
                "block_count": len(members),
                "population": comp_pop,
                "is_single_block_island": len(members) == 1,
                "below_500_unavoidable": comp_pop < 500.0,
                "minimum_source_row_ix": min(members),
            }
        )
    component_summary = pd.DataFrame(component_records)

    # ------------------------------------------------------------------
    # Precision-grid comparisons
    # ------------------------------------------------------------------

    raw_area = shapely.area(geoms).astype(float)
    precision_rows = []
    zone_precision_rows = []
    edge_change_rows = []
    precision_graph_data = {}

    for grid in PRECISION_GRIDS_M:
        tag = grid_tag(grid)
        log(f"Testing precision grid: {grid} m")

        grid_geoms = np.asarray(
            shapely.set_precision(geoms, grid, mode="valid_output"),
            dtype=object,
        )
        grid_valid = shapely.is_valid(grid_geoms)
        grid_empty = shapely.is_empty(grid_geoms)
        grid_area = shapely.area(grid_geoms).astype(float)

        graph, edges = build_eligible_rook_graph(
            gdf, grid_geoms, zone_values, eligible
        )
        assignment, comps = assign_component_ids(graph, zone_values)

        new_edges = edges - raw_edges
        lost_edges = raw_edges - edges
        abs_delta = np.abs(grid_area - raw_area)
        rel_delta_pct = 100.0 * abs_delta / np.maximum(raw_area, 1e-12)

        precision_rows.append(
            {
                "precision_grid_m": grid,
                "rook_edges": len(edges),
                "new_edges_vs_raw": len(new_edges),
                "lost_edges_vs_raw": len(lost_edges),
                "connected_components": len(comps),
                "island_blocks": sum(len(graph[i]) == 0 for i in graph),
                "invalid_after_precision": int((~grid_valid).sum()),
                "empty_after_precision": int(grid_empty.sum()),
                "total_geometry_area_delta_m2": float(
                    grid_area.sum() - raw_area.sum()
                ),
                "sum_absolute_area_change_m2": float(abs_delta.sum()),
                "median_absolute_area_change_m2": float(
                    np.median(abs_delta)
                ),
                "p95_absolute_area_change_m2": float(
                    np.quantile(abs_delta, 0.95)
                ),
                "maximum_absolute_area_change_m2": float(abs_delta.max()),
                "p95_relative_area_change_pct": float(
                    np.quantile(rel_delta_pct, 0.95)
                ),
                "maximum_relative_area_change_pct": float(
                    rel_delta_pct.max()
                ),
            }
        )

        precision_graph_data[grid] = (graph, assignment)

        for zone in sorted(set(zone_values[eligible]), key=lambda x: str(x)):
            nodes = np.flatnonzero(eligible & (zone_values == zone)).tolist()
            zone_edges = sum(len(graph[i]) for i in nodes) // 2
            zone_comps = len({assignment[i] for i in nodes}) if nodes else 0
            zone_islands = sum(len(graph[i]) == 0 for i in nodes)
            zone_new = sum(
                zone_values[i] == zone for i, _ in new_edges
            )
            zone_lost = sum(
                zone_values[i] == zone for i, _ in lost_edges
            )
            zone_precision_rows.append(
                {
                    "zone_id": zone,
                    "precision_grid_m": grid,
                    "eligible_blocks": len(nodes),
                    "rook_edges": zone_edges,
                    "connected_components": zone_comps,
                    "island_blocks": zone_islands,
                    "new_edges_vs_raw": zone_new,
                    "lost_edges_vs_raw": zone_lost,
                }
            )

        for change_type, change_edges in (
            ("new_edge", new_edges),
            ("lost_edge", lost_edges),
        ):
            for i, j in sorted(change_edges):
                raw_shared = float(
                    shapely.length(
                        shapely.intersection(
                            shapely.boundary(geoms[i]),
                            shapely.boundary(geoms[j]),
                        )
                    )
                )
                raw_distance = float(shapely.distance(geoms[i], geoms[j]))
                raw_inter_area = float(
                    shapely.area(shapely.intersection(geoms[i], geoms[j]))
                )
                grid_shared = float(
                    shapely.length(
                        shapely.intersection(
                            shapely.boundary(grid_geoms[i]),
                            shapely.boundary(grid_geoms[j]),
                        )
                    )
                )
                edge_change_rows.append(
                    {
                        "precision_grid_m": grid,
                        "change_type": change_type,
                        "source_row_ix_a": i,
                        "source_row_ix_b": j,
                        "block_id_a": block_ids[i],
                        "block_id_b": block_ids[j],
                        "zone_id": zone_values[i],
                        "raw_shared_boundary_m": raw_shared,
                        "raw_distance_m": raw_distance,
                        "raw_intersection_area_m2": raw_inter_area,
                        "grid_shared_boundary_m": grid_shared,
                    }
                )

    precision_summary = pd.DataFrame(precision_rows)
    zone_precision_summary = pd.DataFrame(zone_precision_rows)
    precision_edge_changes = pd.DataFrame(edge_change_rows)

    # ------------------------------------------------------------------
    # Zone summary
    # ------------------------------------------------------------------

    anomaly_counts = (
        anomalies.groupby(["zone_id", "pair_type"])
        .size()
        .unstack(fill_value=0)
        if len(anomalies)
        else pd.DataFrame()
    )

    zone_rows = []
    for zone in sorted(set(zone_values), key=lambda x: str(x)):
        idx = np.flatnonzero(zone_values == zone)
        eidx = idx[eligible[idx]]
        zone_pop_values = pd.Series(zone_pop[idx]).dropna().unique()
        zone_pop_consistent = len(zone_pop_values) <= 1
        zone_pop_value = (
            float(zone_pop_values[0]) if len(zone_pop_values) == 1 else np.nan
        )

        component_ids = {raw_assignment[i] for i in eidx}
        row = {
            "zone_id": zone,
            "total_blocks": len(idx),
            "eligible_non_open_blocks": len(eidx),
            "open_space_blocks": int(open_flags[idx].sum()),
            "zero_population_eligible_blocks": int(
                (population[eidx] == 0).sum()
            ),
            "zones_5_pop_value": zone_pop_value,
            "zones_5_pop_nunique_nonnull": len(zone_pop_values),
            "zones_5_pop_consistent": zone_pop_consistent,
            "sum_block_population_all": float(population[idx].sum()),
            "sum_block_population_non_open": float(population[eidx].sum()),
            "population_difference_all_minus_zone_pop": (
                float(population[idx].sum() - zone_pop_value)
                if not np.isnan(zone_pop_value)
                else np.nan
            ),
            "processing_branch": (
                "auto_small_zone"
                if zone_pop_consistent
                and not np.isnan(zone_pop_value)
                and zone_pop_value <= 1000.0
                else "iterative_large_zone"
                if zone_pop_consistent and not np.isnan(zone_pop_value)
                else "review_zone_population"
            ),
            "raw_rook_edges": sum(len(raw_graph[i]) for i in eidx) // 2,
            "raw_connected_components": len(component_ids),
            "raw_island_blocks": int(island_mask[eidx].sum()),
            "components_below_500": (
                int(
                    component_summary.loc[
                        component_summary["zone_id"].eq(zone),
                        "below_500_unavoidable",
                    ].sum()
                )
                if len(component_summary)
                else 0
            ),
        }

        if len(anomaly_counts) and zone in anomaly_counts.index:
            for pair_type in anomaly_counts.columns:
                row[f"{pair_type}_pairs"] = int(
                    anomaly_counts.loc[zone, pair_type]
                )
        zone_rows.append(row)

    zone_summary = pd.DataFrame(zone_rows).fillna(
        {
            "point_only_contact_pairs": 0,
            "tiny_overlap_pairs": 0,
            "material_overlap_pairs": 0,
            "near_gap_pairs": 0,
        }
    )

    # ------------------------------------------------------------------
    # Block diagnostic layer
    # ------------------------------------------------------------------

    blocks_diag = gdf.copy()
    blocks_diag["eligible_non_open"] = eligible.astype(np.int8)
    blocks_diag["rook_degree_raw"] = raw_degree
    blocks_diag["component_id_raw"] = [
        raw_assignment.get(i) for i in range(len(gdf))
    ]
    blocks_diag["island_deferred_raw"] = island_mask.astype(np.int8)
    blocks_diag["island_reason_raw"] = island_reasons
    blocks_diag["point_contact_count"] = point_contact_count
    blocks_diag["overlap_pair_count"] = overlap_count
    blocks_diag["near_gap_count"] = near_gap_count
    blocks_diag["nearest_gap_m"] = nearest_gap

    for grid, (graph, assignment) in precision_graph_data.items():
        tag = grid_tag(grid)
        blocks_diag[f"rook_deg_g{tag}"] = [
            len(graph[i]) if i in graph else 0 for i in range(len(gdf))
        ]
        blocks_diag[f"comp_g{tag}"] = [
            assignment.get(i) for i in range(len(gdf))
        ]

    # ------------------------------------------------------------------
    # City summary and output
    # ------------------------------------------------------------------

    city_summary = pd.DataFrame(
        [
            {
                "total_blocks": len(gdf),
                "zones": int(gdf[zone_col].nunique()),
                "open_space_blocks": int(open_flags.sum()),
                "eligible_non_open_blocks": int(eligible.sum()),
                "zero_population_eligible_blocks": int(
                    (eligible & (population == 0)).sum()
                ),
                "raw_rook_edges": len(raw_edges),
                "raw_connected_components": len(raw_components),
                "raw_island_blocks": int(island_mask.sum()),
                "raw_components_below_500": int(
                    component_summary["below_500_unavoidable"].sum()
                ),
                "point_only_contact_pairs": int(
                    (anomalies["pair_type"] == "point_only_contact").sum()
                )
                if len(anomalies)
                else 0,
                "tiny_overlap_pairs": int(
                    (anomalies["pair_type"] == "tiny_overlap").sum()
                )
                if len(anomalies)
                else 0,
                "material_overlap_pairs": int(
                    (anomalies["pair_type"] == "material_overlap").sum()
                )
                if len(anomalies)
                else 0,
                "near_gap_pairs_total": int(total_near_gap_pairs),
                "near_gap_rows_written": int(len(gap_anomalies)),
                "minimum_shared_edge_m": MIN_SHARED_EDGE_M,
                "near_gap_max_m": NEAR_GAP_MAX_M,
            }
        ]
    )

    if os.path.exists(OUT_GPKG):
        os.remove(OUT_GPKG)

    log(f"Writing diagnostic GeoPackage: {OUT_GPKG}")
    pyogrio.write_dataframe(
        blocks_diag,
        OUT_GPKG,
        layer="blocks_topology_diagnostic",
        driver="GPKG",
        use_arrow=USE_ARROW,
        promote_to_multi=True,
    )

    write_table(city_summary, "city_summary.csv", "city_summary")
    write_table(zone_summary, "zone_summary.csv", "zone_summary")
    write_table(
        component_summary,
        "component_summary_raw.csv",
        "component_summary_raw",
    )
    write_table(
        anomalies,
        "topology_anomalies.csv",
        "topology_anomalies",
    )
    write_table(
        precision_summary,
        "precision_summary.csv",
        "precision_summary",
    )
    write_table(
        zone_precision_summary,
        "zone_precision_summary.csv",
        "zone_precision_summary",
    )
    write_table(
        precision_edge_changes,
        "precision_edge_changes.csv",
        "precision_edge_changes",
    )

    log("")
    log("Diagnostics complete.")
    log(f"Output folder: {OUT_FOLDER}")
    log(f"Raw islands: {int(island_mask.sum()):,}")
    log(f"Raw components: {len(raw_components):,}")
    log(f"Anomaly rows: {len(anomalies):,}")
    log("")
    log("Review precision_summary.csv and precision_edge_changes.csv first.")


if __name__ == "__main__":
    try:
        main()
    except Exception:
        log("ERROR:")
        log(traceback.format_exc())
        raise


In [ ]:
import os
import pandas as pd
import pyogrio

gpkg = r"E:\_kigali\_analysis\segments_v2\blocks_5.gpkg"
layer = "blocks_5"

out_folder = r"E:\_kigali\_analysis\segments_v2\block_id_diagnostics"
os.makedirs(out_folder, exist_ok=True)

# Read only block_id. Retain the GeoPackage feature ID so problem
# features can be located in ArcGIS Pro.
df = pyogrio.read_dataframe(
    gpkg,
    layer=layer,
    columns=["block_id"],
    read_geometry=False,
    fid_as_index=True,
)

df = df.rename_axis("gpkg_fid").reset_index()

# ------------------------------------------------------------
# Null and blank values
# ------------------------------------------------------------

null_mask = df["block_id"].isna()

blank_mask = (
    df["block_id"].notna()
    & df["block_id"].astype(str).str.strip().eq("")
)

null_or_blank = df[null_mask | blank_mask].copy()
null_or_blank["issue"] = "blank"
null_or_blank.loc[null_mask, "issue"] = "null"

# ------------------------------------------------------------
# Exact duplicate values
# ------------------------------------------------------------

duplicate_mask = (
    df["block_id"].notna()
    & df["block_id"].duplicated(keep=False)
)

duplicates = df[duplicate_mask].copy()

if not duplicates.empty:
    duplicates["duplicate_count"] = (
        duplicates.groupby("block_id")["block_id"].transform("size")
    )

    duplicates = duplicates.sort_values(
        ["block_id", "gpkg_fid"]
    )

    duplicate_summary = (
        duplicates.groupby("block_id", as_index=False)
        .agg(
            duplicate_count=("gpkg_fid", "size"),
            gpkg_fids=(
                "gpkg_fid",
                lambda x: ", ".join(str(v) for v in x)
            ),
        )
        .sort_values(
            ["duplicate_count", "block_id"],
            ascending=[False, True]
        )
    )
else:
    duplicate_summary = pd.DataFrame(
        columns=["block_id", "duplicate_count", "gpkg_fids"]
    )

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(f"Total features:             {len(df):,}")
print(f"Null block_id values:       {null_mask.sum():,}")
print(f"Blank block_id values:      {blank_mask.sum():,}")
print(f"Rows with duplicate IDs:    {duplicate_mask.sum():,}")
print(f"Distinct duplicated IDs:    {len(duplicate_summary):,}")

if not null_or_blank.empty:
    print("\nNull or blank records:")
    print(null_or_blank.to_string(index=False))

if not duplicate_summary.empty:
    print("\nDuplicated block_id values:")
    print(duplicate_summary.to_string(index=False))

# ------------------------------------------------------------
# Write CSV outputs
# ------------------------------------------------------------

null_path = os.path.join(out_folder, "block_id_null_or_blank.csv")
duplicate_path = os.path.join(out_folder, "block_id_duplicate_rows.csv")
summary_path = os.path.join(out_folder, "block_id_duplicate_summary.csv")

null_or_blank.to_csv(null_path, index=False)
duplicates.to_csv(duplicate_path, index=False)
duplicate_summary.to_csv(summary_path, index=False)

print("\nOutputs:")
print(null_path)
print(duplicate_path)
print(summary_path)